# M13a · Vraisemblance et régression logistique — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m13a_vraisemblance_logistique/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m13a_vraisemblance_logistique/03_notebook.ipynb`, créée par `python tools/start_chapter.py M13a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M13a.1 à M13a.12, M13a.18 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M13a.22 | La log-vraisemblance et le maximum de vraisemblance d'une loi de Bernoulli · `bernoulli_log_likelihood, bernoulli_mle` | 🔨 | ★ | 15 |
|  | M13a.23 | Maximum de vraisemblance catégoriel, avec lissage, et gaussien · `categorical_mle, gaussian_mle` | 🔨 | ★ | 20 |
|  | M13a.24 | La log-loss à partir des scores, sans débordement · `logistic_loss` | 🔨 | ★★ | 20 |
|  | M13a.25 | `scipy.stats`, `scipy.special` et `sklearn.metrics.log_loss` contre tes fonctions | 📦 | ★ | 25 |
| B | M13a.26 | Prédire la log-loss d'un modèle qui répond toujours 0,5, puis toujours la fréquence des 1 | 🔮 | ★ | 10 |
|  | M13a.27 | La variabilité du maximum de vraisemblance d'une loi de Bernoulli d'un échantillon à l'autre | 🔬 | ★★ | 20 |
|  | M13a.28 | L'entropie et l'indice de Gini sur les lois à trois issues : lignes de niveau et maximum au centre | 🎨 | ★ | 25 |
| C | M13a.29 | Chasse à l'erreur : un ln 0, une sigmoïde qui déborde, une variance de maximum de vraisemblance divisée par n − 1 | 🐛 | ★ | 15 |
|  | M13a.30 | Vérifier avec SymPy le maximum de vraisemblance d'une loi géométrique et d'une loi de Poisson | 🛠️ | ★ | 15 |
|  | M13a.31 | Défi : la proportion d'un mélange de deux lois connues par maximum de vraisemblance, à 1e−6 près du maximum (dichotomie ou Newton) | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : maximum de vraisemblance d'une loi de Bernoulli, catégorielle (avec lissage) et normale, log-loss et cross-entropy, sigmoïde, logit et cote, softplus, modèle logistique et sa frontière, hypothèse naïve, a posteriori gaussien, entropie et indice de Gini, information mutuelle, variance intra et inter, sonde linéaire.
- Écrire les cinq fonctions de `mymath/likelihood.py` et les tester, puis comparer avec SciPy et scikit-learn.
- Prédire la log-loss des modèles constants, mesurer la variabilité de $\hat{p}$, dessiner les impuretés, chasser des erreurs de calcul en flottants, vérifier un maximum de vraisemblance avec SymPy, relever un défi de maximum de vraisemblance sans formule.

**Rappel express.** $\ell(\boldsymbol{\theta}) = \sum_i \ln p(x_i \mid \boldsymbol{\theta})$ ; Bernoulli : $\hat{p}$ = la proportion de 1 ; catégorielle : $\hat{p}_k = \frac{n_k}{N}$, lissée $\frac{n_k + \alpha}{N + K\alpha}$ ; normale : $\hat{\mu} = \bar{x}$, $\hat{\sigma}^2 = \frac{1}{n}\sum_i (x_i - \hat{\mu})^2$ ; log-loss $= -\frac{1}{n}\ell = \frac{1}{n}\sum_i \big[s(z_i) - y_iz_i\big]$ ; $\sigma(z) = \frac{1}{1 + e^{-z}}$, $s(z) = \ln(1 + e^z) = \max(z, 0) + \ln(1 + e^{-\lvert z \rvert})$, $-\ln\sigma(z) = s(-z)$ ; cote $e^{z}$, logit $\ln\frac{p}{1 - p}$ ; frontière $\mathbf{w} \cdot \mathbf{x} + b = 0$ ; $H = -\sum_k p_k\log_2 p_k$, $G = 1 - \sum_k p_k^2$, $I(X ; Y) = H(Y) - H(Y \mid X)$ ; variance totale = intra + inter.

## Partie 0 · Vérifier tes exercices papier (✏️ M13a.1 à M13a.12, 🔗 M13a.18)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une liste `[0.15, -2]` ; dans une liste, une fraction s'écrit comme un calcul, sans guillemets : `[2/9, 7/9]` ;
- `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(2, 9) is accepted too, as "2/9"

**Ex M13a.1 — Le maximum de vraisemblance d'une loi de Bernoulli et d'une loi catégorielle**

In [ ]:
# Ex M13a.1: the values you found on paper
answer_M13a_1a = ...  # a) an irreducible fraction, as text
answer_M13a_1b = ...  # b) a number (4 decimals)
answer_M13a_1c = ...  # c) a number (4 decimals)
answer_M13a_1d = ...  # d) a number (4 decimals)
answer_M13a_1e = ...  # e) a list of 4 fractions, written as calculations (2/9) or with 6 decimals
answer_M13a_1f = ...  # f) a list of 4 numbers (4 decimals)
answer_M13a_1g = ...  # g) a list of 4 numbers (4 decimals)
answer_M13a_1h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_1a, answer_M13a_1b, answer_M13a_1c, answer_M13a_1d, answer_M13a_1e, answer_M13a_1f, answer_M13a_1g, answer_M13a_1h]):
    wbm.check(f"M13a.1{letter}", answer)

**Ex M13a.2 — Le maximum de vraisemblance d'une loi normale : μ̂ et σ̂²**

In [ ]:
# Ex M13a.2: the values you found on paper
answer_M13a_2a = ...  # a) a number
answer_M13a_2b = ...  # b) a number
answer_M13a_2c = ...  # c) a number (4 decimals)
answer_M13a_2d = ...  # d) a number (4 decimals)
answer_M13a_2e = ...  # e) a number (4 decimals)
answer_M13a_2f = ...  # f) a number (4 decimals)
answer_M13a_2g = ...  # g) a number (4 decimals)
answer_M13a_2h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M13a_2a, answer_M13a_2b, answer_M13a_2c, answer_M13a_2d, answer_M13a_2e, answer_M13a_2f, answer_M13a_2g, answer_M13a_2h]):
    wbm.check(f"M13a.2{letter}", answer)

**Ex M13a.3 — La log-loss de quatre prédictions, puis la même en cross-entropy**

In [ ]:
# Ex M13a.3: the values you found on paper
answer_M13a_3a = ...  # a) a list of 4 numbers (4 decimals)
answer_M13a_3b = ...  # b) a number (4 decimals)
answer_M13a_3c = ...  # c) a number (3 decimals)
answer_M13a_3d = ...  # d) a number (4 decimals)
answer_M13a_3e = ...  # e) an irreducible fraction, as text
answer_M13a_3f = ...  # f) a number (4 decimals)
answer_M13a_3g = ...  # g) a number (4 decimals)
answer_M13a_3h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_3a, answer_M13a_3b, answer_M13a_3c, answer_M13a_3d, answer_M13a_3e, answer_M13a_3f, answer_M13a_3g, answer_M13a_3h]):
    wbm.check(f"M13a.3{letter}", answer)

**Ex M13a.4 — Le poids d'un seul exemple mal prédit : la log-loss quand p tend vers 0**

In [ ]:
# Ex M13a.4: the values you found on paper
answer_M13a_4a = ...  # a) a number (4 decimals)
answer_M13a_4b = ...  # b) a number (4 decimals)
answer_M13a_4c = ...  # c) a number (4 decimals)
answer_M13a_4d = ...  # d) a number (2 decimals)
answer_M13a_4e = ...  # e) a number (4 decimals)
answer_M13a_4f = ...  # f) a number (4 decimals)
answer_M13a_4g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M13a_4a, answer_M13a_4b, answer_M13a_4c, answer_M13a_4d, answer_M13a_4e, answer_M13a_4f, answer_M13a_4g]):
    wbm.check(f"M13a.4{letter}", answer)

**Ex M13a.5 — La sigmoïde : valeurs, logit, cote et symétrie σ(−z) = 1 − σ(z)**

In [ ]:
# Ex M13a.5: the values you found on paper
answer_M13a_5a = ...  # a) a number (4 decimals)
answer_M13a_5b = ...  # b) a number (4 decimals)
answer_M13a_5c = ...  # c) a number
answer_M13a_5d = ...  # d) a number (4 decimals)
answer_M13a_5e = ...  # e) an irreducible fraction, as text
answer_M13a_5f = ...  # f) a number (4 decimals)
answer_M13a_5g = ...  # g) an irreducible fraction, as text
answer_M13a_5h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M13a_5a, answer_M13a_5b, answer_M13a_5c, answer_M13a_5d, answer_M13a_5e, answer_M13a_5f, answer_M13a_5g, answer_M13a_5h]):
    wbm.check(f"M13a.5{letter}", answer)

**Ex M13a.6 — La softplus : valeurs, asymptotes et comportement pour |z| grand**

In [ ]:
# Ex M13a.6: the values you found on paper
answer_M13a_6a = ...  # a) a number (4 decimals)
answer_M13a_6b = ...  # b) a number (4 decimals)
answer_M13a_6c = ...  # c) a number
answer_M13a_6d = ...  # d) a number (6 decimals)
answer_M13a_6e = ...  # e) a number (4 decimals)
answer_M13a_6f = ...  # f) a number (4 decimals)
answer_M13a_6g = ...  # g) True or False
answer_M13a_6h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_6a, answer_M13a_6b, answer_M13a_6c, answer_M13a_6d, answer_M13a_6e, answer_M13a_6f, answer_M13a_6g, answer_M13a_6h]):
    wbm.check(f"M13a.6{letter}", answer)

**Ex M13a.7 — Un modèle logistique à deux features : probabilités, cote et frontière de décision**

In [ ]:
# Ex M13a.7: the values you found on paper
answer_M13a_7a = ...  # a) a number (4 decimals)
answer_M13a_7b = ...  # b) a number (4 decimals)
answer_M13a_7c = ...  # c) a number (4 decimals)
answer_M13a_7d = ...  # d) a number (4 decimals)
answer_M13a_7e = ...  # e) a number (4 decimals)
answer_M13a_7f = ...  # f) a number (4 decimals)
answer_M13a_7g = ...  # g) a number (4 decimals)
answer_M13a_7h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_7a, answer_M13a_7b, answer_M13a_7c, answer_M13a_7d, answer_M13a_7e, answer_M13a_7f, answer_M13a_7g, answer_M13a_7h]):
    wbm.check(f"M13a.7{letter}", answer)

**Ex M13a.8 — L'hypothèse naïve à la main : une loi jointe factorisée, comparée à la vraie**

In [ ]:
# Ex M13a.8: the values you found on paper
answer_M13a_8a = ...  # a) an irreducible fraction, as text
answer_M13a_8b = ...  # b) an irreducible fraction, as text
answer_M13a_8c = ...  # c) an irreducible fraction, as text
answer_M13a_8d = ...  # d) a number (4 decimals)
answer_M13a_8e = ...  # e) a number (4 decimals)
answer_M13a_8f = ...  # f) a number (4 decimals)
answer_M13a_8g = ...  # g) an integer
answer_M13a_8h = ...  # h) a list of 2 integers

for letter, answer in zip("abcdefgh", [answer_M13a_8a, answer_M13a_8b, answer_M13a_8c, answer_M13a_8d, answer_M13a_8e, answer_M13a_8f, answer_M13a_8g, answer_M13a_8h]):
    wbm.check(f"M13a.8{letter}", answer)

**Ex M13a.9 — Deux classes gaussiennes de même variance : l'a posteriori est une sigmoïde d'un score affine**

In [ ]:
# Ex M13a.9: the values you found on paper
answer_M13a_9a = ...  # a) a number
answer_M13a_9b = ...  # b) a number (4 decimals)
answer_M13a_9c = ...  # c) a number (4 decimals)
answer_M13a_9d = ...  # d) a number (4 decimals)
answer_M13a_9e = ...  # e) a number (3 decimals)
answer_M13a_9f = ...  # f) a number
answer_M13a_9g = ...  # g) a number (4 decimals)
answer_M13a_9h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_9a, answer_M13a_9b, answer_M13a_9c, answer_M13a_9d, answer_M13a_9e, answer_M13a_9f, answer_M13a_9g, answer_M13a_9h]):
    wbm.check(f"M13a.9{letter}", answer)

**Ex M13a.10 — L'entropie et l'indice de Gini de quelques distributions**

In [ ]:
# Ex M13a.10: the values you found on paper
answer_M13a_10a = ...  # a) a number (4 decimals)
answer_M13a_10b = ...  # b) a number (4 decimals)
answer_M13a_10c = ...  # c) a number (4 decimals)
answer_M13a_10d = ...  # d) a number (4 decimals)
answer_M13a_10e = ...  # e) an irreducible fraction, as text
answer_M13a_10f = ...  # f) an irreducible fraction, as text
answer_M13a_10g = ...  # g) a number (4 decimals)
answer_M13a_10h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M13a_10a, answer_M13a_10b, answer_M13a_10c, answer_M13a_10d, answer_M13a_10e, answer_M13a_10f, answer_M13a_10g, answer_M13a_10h]):
    wbm.check(f"M13a.10{letter}", answer)

**Ex M13a.11 — L'information mutuelle d'un tableau de contingence, et la baisse d'entropie qu'elle mesure**

In [ ]:
# Ex M13a.11: the values you found on paper
answer_M13a_11a = ...  # a) an irreducible fraction, as text
answer_M13a_11b = ...  # b) a number (4 decimals)
answer_M13a_11c = ...  # c) a number (4 decimals)
answer_M13a_11d = ...  # d) a number (4 decimals)
answer_M13a_11e = ...  # e) a number (4 decimals)
answer_M13a_11f = ...  # f) a number (4 decimals)
answer_M13a_11g = ...  # g) a number (4 decimals)
answer_M13a_11h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M13a_11a, answer_M13a_11b, answer_M13a_11c, answer_M13a_11d, answer_M13a_11e, answer_M13a_11f, answer_M13a_11g, answer_M13a_11h]):
    wbm.check(f"M13a.11{letter}", answer)

**Ex M13a.12 — La variance d'un groupe coupé en deux : variance intra plus variance inter**

In [ ]:
# Ex M13a.12: the values you found on paper
answer_M13a_12a = ...  # a) a number (4 decimals)
answer_M13a_12b = ...  # b) a number (4 decimals)
answer_M13a_12c = ...  # c) a list of 2 numbers
answer_M13a_12d = ...  # d) a list of 2 numbers (4 decimals)
answer_M13a_12e = ...  # e) a number (4 decimals)
answer_M13a_12f = ...  # f) a number (4 decimals)
answer_M13a_12g = ...  # g) True or False
answer_M13a_12h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M13a_12a, answer_M13a_12b, answer_M13a_12c, answer_M13a_12d, answer_M13a_12e, answer_M13a_12f, answer_M13a_12g, answer_M13a_12h]):
    wbm.check(f"M13a.12{letter}", answer)

**Ex M13a.18 — Pont : une sonde linéaire, un modèle logistique en NumPy sur un batch de représentations figées, jusqu'à la log-loss**

In [ ]:
# Ex M13a.18: the values you found on paper
answer_M13a_18a = ...  # a) a list of 6 numbers (4 decimals)
answer_M13a_18b = ...  # b) a list of 6 numbers (4 decimals)
answer_M13a_18c = ...  # c) a number (4 decimals)
answer_M13a_18d = ...  # d) an irreducible fraction, as text
answer_M13a_18e = ...  # e) a number (4 decimals)
answer_M13a_18f = ...  # f) a number (4 decimals)
answer_M13a_18g = ...  # g) an integer
answer_M13a_18h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M13a_18a, answer_M13a_18b, answer_M13a_18c, answer_M13a_18d, answer_M13a_18e, answer_M13a_18f, answer_M13a_18g, answer_M13a_18h]):
    wbm.check(f"M13a.18{letter}", answer)

## Partie A · Ta librairie : log-vraisemblance, maximum de vraisemblance et log-loss ; SciPy et scikit-learn

Fiche §1 à §3. Les cinq fonctions de `mymath/likelihood.py`, puis les mêmes calculs avec `scipy.stats`, `scipy.special` et `sklearn.metrics`.

In [ ]:
# Tools of the notebook (parts A to C)
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy

from wbm.nbtools import checked, drawn, filled, run_pytest_group, verdict


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M13a.22 — La log-vraisemblance et le maximum de vraisemblance d'une loi de Bernoulli · `bernoulli_log_likelihood, bernoulli_mle` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la log-vraisemblance de labels de Bernoulli, avec ses deux cas limites (un terme $0 \ln 0$, un label impossible), et son maximum.  
**Prérequis :** fiche §1 · **mymath :** `likelihood.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/likelihood.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy et le module `math` sont permis ; SciPy et scikit-learn sont interdits (les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais les tableaux reçus : travaille sur une copie, `np.array(y, dtype=float)`. Les nombres se rendent en `float` de Python, les vecteurs en tableaux NumPy de flottants. Les logarithmes sont népériens, et aucun calcul ne doit produire `nan` ni d'avertissement (`RuntimeWarning`) : un $\ln 0$ se traite à part.

Écris, en suivant les docstrings (fiche §1) :
- `bernoulli_log_likelihood(y, p)` : $\sum_i \big[y_i\ln p_i + (1 - y_i)\ln(1 - p_i)\big]$, avec une seule probabilité `p` pour tous les labels ou une par label. Les contrôles d'abord (`ValueError` si `y` est vide ou contient autre chose que des 0 et des 1, si `p` n'a pas la bonne longueur, une liste d'une seule valeur pour plusieurs labels comprise, ou a une valeur hors de $[0, 1]$, `nan` compris : `nan < 0` et `nan > 1` valent tous deux `False`, donc teste plutôt que chaque valeur **est** dans $[0, 1]$). Puis, pour chaque label, le $\ln$ de la probabilité **du label observé** seulement : $p_i$ pour un 1, $1 - p_i$ pour un 0. Ainsi un terme $0 \ln 0$ ne se calcule jamais (il compte pour 0), et un label de probabilité 0 rend `-math.inf`, sans avertissement ;
- `bernoulli_mle(y)` : la proportion de 1, avec les mêmes contrôles sur `y`.

La vérification calcule, avec les 50 labels `Y_22` de la cellule fournie :
- a) `bernoulli_log_likelihood(Y_22, 0.2)` (6 décimales) ;
- b) `bernoulli_mle(Y_22)` (6 décimales) ;
- c) la log-vraisemblance en ce maximum, `bernoulli_log_likelihood(Y_22, bernoulli_mle(Y_22))` (6 décimales) ;
- d) vrai ou faux : c) est au moins égale (à $10^{-12}$ près) à `bernoulli_log_likelihood(Y_22, q)` pour les 99 valeurs `q` de `np.linspace(0.01, 0.99, 99)` ;
- e) `bernoulli_log_likelihood([0, 1, 1, 0], [0.0, 0.5, 1.0, 0.2])` (6 décimales) : deux des labels sont certains ;
- f) `bernoulli_log_likelihood([0, 1, 1, 0], [0.3, 0.8, 0.0, 0.1])` : un label observé avait reçu la probabilité 0 ;
- g) le nom de l'erreur levée par `bernoulli_log_likelihood([0, 1, 2, 1], 0.5)`.

Dans ta copie : retrouve b) et c) à la main, à partir du nombre de 1 de `Y_22` (`Y_22.sum()`). Combien de fois la vraisemblance $L(\hat{p})$ est-elle plus grande que $L(0{,}2)$ ? Pourquoi la formule $y\ln p + (1 - y)\ln(1 - p)$, écrite telle quelle avec NumPy, donne-t-elle `nan` pour e), et que vaut chacun des quatre termes en vérité ? Que dit la valeur de f) sur le modèle qui a donné ces probabilités ?

In [ ]:
Y_22 = (np.random.default_rng(4122).random(50) < 0.35).astype(int)   # 50 synthetic labels, 0 or 1

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13a")  # reload your saved file

In [ ]:
with checked("M13a.22a"):
    wbm.check("M13a.22a", mymath.likelihood.bernoulli_log_likelihood(Y_22, 0.2), computed=True)
with checked("M13a.22b"):
    wbm.check("M13a.22b", mymath.likelihood.bernoulli_mle(Y_22), computed=True)
with checked("M13a.22c"):
    wbm.check("M13a.22c", mymath.likelihood.bernoulli_log_likelihood(Y_22, mymath.likelihood.bernoulli_mle(Y_22)),
              computed=True)
with checked("M13a.22d"):
    best_22 = mymath.likelihood.bernoulli_log_likelihood(Y_22, mymath.likelihood.bernoulli_mle(Y_22))
    wbm.check("M13a.22d", all(best_22 >= mymath.likelihood.bernoulli_log_likelihood(Y_22, float(q)) - 1e-12
                              for q in np.linspace(0.01, 0.99, 99)), computed=True)
with checked("M13a.22e"):
    wbm.check("M13a.22e", mymath.likelihood.bernoulli_log_likelihood([0, 1, 1, 0], [0.0, 0.5, 1.0, 0.2]), computed=True)
with checked("M13a.22f"):
    wbm.check("M13a.22f", mymath.likelihood.bernoulli_log_likelihood([0, 1, 1, 0], [0.3, 0.8, 0.0, 0.1]), computed=True)
with checked("M13a.22g"):
    wbm.check("M13a.22g", raised(mymath.likelihood.bernoulli_log_likelihood, [0, 1, 2, 1], 0.5), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13a_likelihood.py", "test_bernoulli_log_likelihood or test_bernoulli_mle or test_likelihood_uses_")

### Ex M13a.23 — Maximum de vraisemblance catégoriel, avec lissage, et gaussien · `categorical_mle, gaussian_mle` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le maximum de vraisemblance d'une loi catégorielle, avec lissage, et celui d'une loi normale.  
**Prérequis :** fiche §1 et §5 · **mymath :** `likelihood.py` · **Parcours :** C, E

Écris, en suivant les docstrings :
- `categorical_mle(counts, alpha=0.0)` (fiche §1) : `ValueError` si les comptages sont vides, pas 1-D, ou ont une valeur négative ou `nan` (même remarque qu'en M13a.22), si `alpha` est négatif, ou si tous les comptages sont nuls avec `alpha = 0` ; sinon $\frac{n_k + \alpha}{N + K\alpha}$ (le maximum de vraisemblance $\frac{n_k}{N}$ pour $\alpha = 0$) ;
- `gaussian_mle(x)` (fiche §1) : `ValueError` si l'échantillon est vide ou pas 1-D ; sinon le couple $(\hat{\mu}, \hat{\sigma}^2)$, avec la variance **en $n$**, calculée sur les écarts $x_i - \hat{\mu}$.

La vérification calcule, avec les comptages `COUNTS_23` des six faces d'un dé truqué et l'échantillon `X_23` :
- a) `categorical_mle(COUNTS_23)` (une liste, 6 décimales) ;
- b) `categorical_mle(COUNTS_23, alpha=1)` (6 décimales) ;
- c) `categorical_mle(COUNTS_23, alpha=0.5)` (6 décimales) ;
- d) `gaussian_mle(X_23)`, sous la forme d'une liste $[\hat{\mu}, \hat{\sigma}^2]$ (6 décimales) ;
- e) le rapport `gaussian_mle(X_23)[1] / np.var(X_23, ddof=1)` (6 décimales) ;
- f) le nom de l'erreur levée par `categorical_mle([0, 0, 0, 0, 0, 0])`.

Dans ta copie : retrouve à la main les estimations de la face 3 en a), b) et c). Pourquoi celle de la face 4 est-elle la même dans les trois cas ? Vers quelle loi le lissage tire-t-il les estimations quand $\alpha$ grandit ? Retrouve e) sans rien calculer sur `X_23`. Quelle log-vraisemblance aurait un 67ᵉ lancer tombé sur la face 3, selon a) et selon b) ?

In [ ]:
COUNTS_23 = [9, 15, 0, 11, 7, 24]      # 66 throws of a loaded die: how many times each face 1 to 6 came up
X_23 = np.random.default_rng(4123).normal(12.0, 3.0, size=40)   # 40 synthetic measures

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13a")  # reload your saved file

In [ ]:
with checked("M13a.23a"):
    wbm.check("M13a.23a", np.asarray(mymath.likelihood.categorical_mle(COUNTS_23)).tolist(), computed=True)
with checked("M13a.23b"):
    wbm.check("M13a.23b", np.asarray(mymath.likelihood.categorical_mle(COUNTS_23, alpha=1)).tolist(), computed=True)
with checked("M13a.23c"):
    wbm.check("M13a.23c", np.asarray(mymath.likelihood.categorical_mle(COUNTS_23, alpha=0.5)).tolist(), computed=True)
with checked("M13a.23d"):
    wbm.check("M13a.23d", list(mymath.likelihood.gaussian_mle(X_23)), computed=True)
with checked("M13a.23e"):
    wbm.check("M13a.23e", mymath.likelihood.gaussian_mle(X_23)[1] / np.var(X_23, ddof=1), computed=True)
with checked("M13a.23f"):
    wbm.check("M13a.23f", raised(mymath.likelihood.categorical_mle, [0, 0, 0, 0, 0, 0]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13a_likelihood.py", "test_categorical_mle or test_gaussian_mle or test_likelihood_uses_")

### Ex M13a.24 — La log-loss à partir des scores, sans débordement · `logistic_loss` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la log-loss d'un modèle logistique directement à partir des scores, juste et sans débordement pour des scores immenses.  
**Prérequis :** fiche §2 et §3 · **mymath :** `likelihood.py` · **Parcours :** C, E

Écris `logistic_loss(z, y)` (fiche §3), en suivant la docstring : `ValueError` si les longueurs diffèrent, si `y` est vide ou contient autre chose que des 0 et des 1 ; puis la moyenne des $s(z_i) - y_i z_i$, avec la softplus calculée **sans débordement** : $s(z) = \max(z, 0) + \ln(1 + e^{-\lvert z \rvert})$ (`np.maximum`, `np.log1p`). L'exponentielle ne reçoit ainsi jamais un nombre positif.

La vérification calcule :
- a) `logistic_loss(Z_24, Y_24)` pour les 30 scores modérés de la cellule fournie (6 décimales) : elle est comparée à `sklearn.metrics.log_loss(Y_24, σ(Z_24))` ;
- b) `logistic_loss([900.0, -900.0, 900.0, -900.0], [0, 1, 1, 0])` (6 décimales) ;
- c) vrai ou faux : le calcul de b) ne produit aucun avertissement (`RuntimeWarning`) ;
- d) `logistic_loss([900.0, -900.0], [1, 0])` (6 décimales) : deux prédictions sûres et justes ;
- e) le nom de l'erreur levée par `logistic_loss([0.3, -0.7, 1.9], [1, 0])`.

Dans ta copie : retrouve b) à la main, terme par terme, avec l'écriture stable. Pourquoi `np.log(1 + np.exp(z))` rend-il `inf` (avec un avertissement) pour $z = 900$, et pourquoi la formule « avec $\hat{p} = \sigma(z)$ » perd-elle tout bien avant ($\sigma(45)$ vaut exactement 1 en flottants) ? Que donnerait la formule naïve pour d) ? Compare a) à $\ln 2$, la log-loss d'un modèle qui répond toujours 0,5.

In [ ]:
rng_24 = np.random.default_rng(4124)
Z_24 = rng_24.normal(0.5, 2.0, size=30)                             # 30 moderate scores
Y_24 = (rng_24.random(30) < 1 / (1 + np.exp(-Z_24))).astype(int)   # their labels, drawn with the probabilities σ(z)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13a")  # reload your saved file

In [ ]:
with checked("M13a.24a"):
    wbm.check("M13a.24a", mymath.likelihood.logistic_loss(Z_24, Y_24), computed=True)
with checked("M13a.24b"):
    wbm.check("M13a.24b", mymath.likelihood.logistic_loss([900.0, -900.0, 900.0, -900.0], [0, 1, 1, 0]), computed=True)
with checked("M13a.24c"):
    with warnings.catch_warnings():
        warnings.simplefilter("error", RuntimeWarning)       # a warning becomes an error, caught just below
        try:
            mymath.likelihood.logistic_loss([900.0, -900.0, 900.0, -900.0], [0, 1, 1, 0])
            quiet_24 = True
        except RuntimeWarning:
            quiet_24 = False
    wbm.check("M13a.24c", quiet_24, computed=True)
with checked("M13a.24d"):
    wbm.check("M13a.24d", mymath.likelihood.logistic_loss([900.0, -900.0], [1, 0]), computed=True)
with checked("M13a.24e"):
    wbm.check("M13a.24e", raised(mymath.likelihood.logistic_loss, [0.3, -0.7, 1.9], [1, 0]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13a_likelihood.py", "test_logistic_loss or test_likelihood_uses_")

### Ex M13a.25 — `scipy.stats`, `scipy.special` et `sklearn.metrics.log_loss` contre tes fonctions 📦 ★ ⏱️ 25 min
**Objectif :** utiliser `scipy.stats.bernoulli`, `scipy.stats.norm.fit`, `scipy.special.xlogy`, `scipy.special.log_expit` et `sklearn.metrics.log_loss`, lire ce qu'elles rendent, et comparer avec ta librairie.  
**Prérequis :** fiche §1 à §3 ; Ex M13a.22 à M13a.24 · **Parcours :** C, E

En pratique, ces calculs sont tout faits :
- `stats.bernoulli.logpmf(y, p)` rend le tableau des $\ln P(Y = y_i)$, un par label (`p` peut être un tableau) ;
- `stats.norm.fit(x)` rend le couple $(\hat{\mu}, \hat{\sigma})$ du maximum de vraisemblance d'une loi normale : l'**écart-type**, pas la variance ;
- `special.xlogy(a, b)` calcule $a\ln b$, avec la convention $0 \ln 0 = 0$, sans avertissement ;
- `special.log_expit(z)` calcule $\ln\sigma(z)$ sans débordement ;
- `log_loss(y, p)` (scikit-learn) rend la log-loss moyenne, en nats, à partir des **probabilités**, qu'elle écrête loin de 0 et de 1.

Avec les données de la cellule fournie, calcule :
- a) `stats.bernoulli.logpmf(Y_25, P_25).sum()` (6 décimales) ;
- b) vrai ou faux : ta fonction `bernoulli_log_likelihood(Y_25, P_25)` est égale à a), à $10^{-10}$ près relativement ;
- c) les deux nombres rendus par `stats.norm.fit(X_25)` (une liste, 6 décimales) ;
- d) vrai ou faux : ta fonction `gaussian_mle(X_25)` rend la même moyenne que c) et, pour variance, le **carré** du second nombre de c), à $10^{-10}$ près relativement ;
- e) la log-vraisemblance catégorielle des comptages `COUNTS_25` en leur maximum, `special.xlogy(COUNTS_25, COUNTS_25 / COUNTS_25.sum()).sum()` (6 décimales) ;
- f) vrai ou faux : ta fonction `logistic_loss(Z_25, Y_25)` est égale à la moyenne de `-special.log_expit(np.where(Y_25 == 1, Z_25, -Z_25))`, à $10^{-12}$ près relativement ;
- g) `log_loss([1], [0.0], labels=[0, 1])` : la log-loss d'un seul exemple, de label 1, auquel le modèle a donné la probabilité 0 (4 décimales). Ne devine pas : **exécute-la** ;
- h) l'entier $k$ tel que $e^{-g} = 2^{-k}$ : la probabilité que `log_loss` a utilisée à la place de 0.

Dans ta copie : pourquoi le second nombre de c) n'est-il pas une variance, et que rendrait `np.std(X_25, ddof=1)` ? Que donne `np.sum(COUNTS_25 * np.log(COUNTS_25 / COUNTS_25.sum()))`, et pourquoi ? Que vaut `np.finfo(float).eps`, et quel lien avec h) ? En M13a.4 e), une bibliothèque écrêtait à $10^{-15}$ : pourquoi deux bibliothèques peuvent-elles rendre des log-loss différentes pour les mêmes prédictions ? Pourquoi `special.log_expit(z)` plutôt que `np.log(special.expit(z))` ?

In [ ]:
from scipy import special, stats
from sklearn.metrics import log_loss

rng_25 = np.random.default_rng(4125)
Y_25 = (rng_25.random(40) < 0.6).astype(int)                 # 40 labels
P_25 = np.round(rng_25.uniform(0.05, 0.95, size=40), 2)     # the probabilities of label 1 given by a model
Z_25 = rng_25.normal(0.0, 4.0, size=40)                     # the scores of another model, for the same labels
X_25 = rng_25.normal(-4.0, 1.5, size=25)                    # a sample of real values
COUNTS_25 = np.array([14, 0, 7, 3, 6])                      # the counts of five categories

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13a")
answer_M13a_25a = ...  # a) a number (6 decimals)
answer_M13a_25c = ...  # c) a list of 2 numbers (6 decimals)
answer_M13a_25e = ...  # e) a number (6 decimals)
answer_M13a_25g = ...  # g) a number (4 decimals)
answer_M13a_25h = ...  # h) an integer
answer_M13a_25b = answer_M13a_25d = answer_M13a_25f = ...
# Each block stands alone: compute inside it what it needs, so that "Run all" goes on when a function is not written.
with wbm.attempt("M13a.25b"):        # your bernoulli_log_likelihood is called here: ⏳ while it is not written
    answer_M13a_25b = ...  # b) True or False
with wbm.attempt("M13a.25d"):        # your gaussian_mle
    answer_M13a_25d = ...  # d) True or False
with wbm.attempt("M13a.25f"):        # your logistic_loss
    answer_M13a_25f = ...  # f) True or False

In [ ]:
for letter, answer in zip("abcdefgh", [answer_M13a_25a, answer_M13a_25b, answer_M13a_25c, answer_M13a_25d,
                                        answer_M13a_25e, answer_M13a_25f, answer_M13a_25g, answer_M13a_25h]):
    wbm.check(f"M13a.25{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §1, §2 et §6. La log-loss des modèles constants, la variabilité de $\hat{p}$ d'un échantillon à l'autre, et les deux impuretés sur le triangle des lois à trois issues.

### Ex M13a.26 — Prédire la log-loss d'un modèle qui répond toujours 0,5, puis toujours la fréquence des 1 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la log-loss de modèles qui ne regardent pas les features, et lequel d'entre eux est le meilleur, puis le vérifier.  
**Prérequis :** fiche §2 · **Parcours :** C, E

La cellule fournie fabrique 1 000 labels synthétiques `Y_26`, dont exactement 150 valent 1 (une fréquence de 0,15), et 1 000 labels `ONES_26`, tous à 1. Un modèle **constant** donne la même probabilité $q$ du label 1 à tous les exemples, sans regarder leurs features. Pour chaque question, choisis le numéro de la bonne réponse :
- a) la log-loss (en nats) du modèle qui répond toujours $q = 0{,}5$, sur `Y_26` : (1) environ 0,22 ; (2) environ 0,42 ; (3) environ 0,69 ; (4) environ 1,90 ;
- b) la log-loss du modèle qui répond toujours $q = 0{,}15$, sur `Y_26` : mêmes choix ;
- c) la log-loss du modèle qui répond toujours $q = 0{,}5$, sur `ONES_26` : (1) 0 ; (2) environ 0,35 ; (3) environ 0,69 ; (4) $+\infty$ ;
- d) parmi les modèles constants $q = 0{,}05$, $0{,}15$, $0{,}35$ et $0{,}5$, celui qui a la plus petite log-loss sur `Y_26` : (1) 0,05 ; (2) 0,15 ; (3) 0,35 ; (4) 0,5.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

In [ ]:
Y_26 = np.zeros(1000, dtype=int)
Y_26[np.random.default_rng(4126).permutation(1000)[:150]] = 1   # 1 000 synthetic labels, exactly 150 of them 1
ONES_26 = np.ones(1000, dtype=int)                              # 1 000 labels, all 1

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M13a_26a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M13a_26b = ...  # b) 1, 2, 3 or 4
prediction_M13a_26c = ...  # c) 1, 2, 3 or 4
prediction_M13a_26d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M13a_26a, prediction_M13a_26b, prediction_M13a_26c,
                                      prediction_M13a_26d]):
    wbm.check(f"M13a.26{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M13a_26a, prediction_M13a_26b, prediction_M13a_26c, prediction_M13a_26d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    def constant_log_loss(labels, q):
        """The log-loss (in nats) of the model that gives the same probability q of label 1 to every example."""
        labels = np.asarray(labels, dtype=float)
        return float(-np.mean(labels * np.log(q) + (1 - labels) * np.log(1 - q)))


    print(f"a) toujours 0,5 sur Y_26 : {constant_log_loss(Y_26, 0.5):.4f}".replace(".", ","))
    print(f"b) toujours 0,15 sur Y_26 : {constant_log_loss(Y_26, 0.15):.4f}".replace(".", ","))
    print(f"c) toujours 0,5 sur des labels tous à 1 : {constant_log_loss(ONES_26, 0.5):.4f}".replace(".", ","))
    print("d) " + " ; ".join(f"toujours {q} : {constant_log_loss(Y_26, q):.4f}" for q in (0.05, 0.15, 0.35, 0.5)).replace(".", ","))
    qs = np.linspace(0.01, 0.99, 197)
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.plot(qs, [constant_log_loss(Y_26, q) for q in qs], label="labels Y_26 (15 % de 1)")
    ax.plot(qs, [constant_log_loss(ONES_26, q) for q in qs], label="labels tous à 1")
    ax.axvline(0.15, color="0.6", lw=0.8, ls="--")
    ax.set_xlabel("probabilité q annoncée à chaque exemple")
    ax.set_ylabel("log-loss (nats)")
    ax.set_ylim(0, 3)
    ax.legend()
    plt.show()

**Après l'expérience**, dans ta copie : pourquoi a) et c) sont-elles égales, et pourquoi la log-loss du modèle 0,5 ne dépend-elle pas des labels ? Que vaut b) en termes d'entropie (fiche §2) ? Pourquoi aucun modèle constant ne fait-il mieux que b) sur `Y_26`, et que doit donc battre un modèle qui regarde les features ? Sur les labels tous à 1, pourquoi la courbe n'a-t-elle pas de minimum dans $]0, 1[$ ?

### Ex M13a.27 — La variabilité du maximum de vraisemblance d'une loi de Bernoulli d'un échantillon à l'autre 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer de combien l'estimation $\hat{p}$ varie d'un échantillon à l'autre, comparer à la formule $\sqrt{p(1 - p)/n}$, et voir l'effet de la taille de l'échantillon.  
**Prérequis :** fiche §1 ; M02c · **Parcours :** C

Une loi de Bernoulli de paramètre $p = 0{,}3$ (connu ici, puisque c'est toi qui tires les données). Pour chaque taille $n$ de `SIZES_27`, tire 2 000 échantillons de $n$ labels, calcule le maximum de vraisemblance $\hat{p}$ de chacun (la proportion de 1), et regarde comment ces 2 000 valeurs se dispersent. Écris :
- `phat_samples(rng, n)` : les 2 000 valeurs de $\hat{p}$ pour des échantillons de taille $n$, en tirant les labels par `rng.random((N_SAMPLES_27, n)) < P_27` (un échantillon par ligne), puis en prenant la moyenne de chaque ligne ;
- `spread_table(seed=4127)` : le dictionnaire `{n: les 2 000 valeurs de p̂}` pour $n$ = 25, 100 et 400, **dans cet ordre**, avec **un seul** générateur `np.random.default_rng(seed)`, créé une fois (c'est ce qui rend tes nombres égaux à ceux de la vérification).

Puis calcule :
- a) les écarts-types des 2 000 valeurs de $\hat{p}$ pour $n$ = 25, 100 et 400, avec `np.std(..., ddof=1)` (une liste, 5 décimales) ;
- b) leurs valeurs théoriques $\sqrt{p(1 - p)/n}$ (une liste, 5 décimales) ;
- c) vrai ou faux : chaque écart-type mesuré est à moins de 5 % de sa valeur théorique ;
- d) les deux rapports des écarts-types mesurés, $n = 25$ sur $n = 100$ puis $n = 100$ sur $n = 400$ (une liste, 2 décimales) ;
- e) l'entier par lequel l'écart-type théorique est divisé quand $n$ est multiplié par 4.

Une cellule trace ensuite les trois histogrammes, avec la densité de la loi normale $\mathcal{N}\big(p, \frac{p(1 - p)}{n}\big)$.

Dans ta copie : d'où vient $\sqrt{p(1 - p)/n}$ (M02c : la variance d'une moyenne de $n$ variables indépendantes) ? Combien de labels faudrait-il pour que l'écart-type de $\hat{p}$ tombe sous 0,01 ? Pourquoi les histogrammes ressemblent-ils à des cloches de plus en plus étroites, et quel lien avec la figure de la fiche §1 (une log-vraisemblance plus creuse quand $n$ grandit) ? Que dire d'une proportion estimée sur 25 exemples à « 0,32 » ?

In [ ]:
P_27 = 0.3                    # the true parameter of the Bernoulli law
SIZES_27 = (25, 100, 400)     # the sizes n of the samples
N_SAMPLES_27 = 2000           # the number of samples for each n

In [ ]:
def phat_samples(rng, n):
    """The N_SAMPLES_27 maximum likelihood estimates p̂ of samples of n labels of a Bernoulli law of parameter P_27:
    the labels drawn as rng.random((N_SAMPLES_27, n)) < P_27 (one sample per row), then the mean of each row."""
    raise NotImplementedError("rng.random((N_SAMPLES_27, n)) < P_27, puis la moyenne de chaque ligne")


def spread_table(seed=4127):
    """{n: the array of the N_SAMPLES_27 values of p̂} for n in SIZES_27, in this order, with ONE generator
    np.random.default_rng(seed), created once."""
    raise NotImplementedError("un seul générateur, puis phat_samples(rng, n) pour n = 25, 100 et 400, dans cet ordre")


answer_M13a_27a = answer_M13a_27c = answer_M13a_27d = ...
with wbm.attempt("M13a.27a"):        # your spread_table is called here: ⏳ while it is not written
    table_27 = spread_table()
    answer_M13a_27a = ...  # a) a list of 3 numbers (5 decimals)
    answer_M13a_27c = ...  # c) True or False
    answer_M13a_27d = ...  # d) a list of 2 numbers (2 decimals)
answer_M13a_27b = ...  # b) a list of 3 numbers (5 decimals)
answer_M13a_27e = ...  # e) an integer

In [ ]:
for letter, answer in zip("abcde", [answer_M13a_27a, answer_M13a_27b, answer_M13a_27c, answer_M13a_27d,
                                     answer_M13a_27e]):
    wbm.check(f"M13a.27{letter}", answer)

with checked("M13a.27 figure"):
    table_fig_27 = spread_table()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    try:
        grid_27 = np.linspace(0.0, 0.65, 300)
        for ax, n in zip(axes, SIZES_27):
            sd = math.sqrt(P_27 * (1 - P_27) / n)
            ax.hist(table_fig_27[n], bins=np.arange(-0.5, n + 1.5, max(1, n // 100)) / n, density=True, alpha=0.6,
                    label="les 2 000 p̂")
            ax.plot(grid_27, np.exp(-(grid_27 - P_27) ** 2 / (2 * sd ** 2)) / (sd * math.sqrt(2 * math.pi)), color="C3",
                    label="N(p, p(1 − p)/n)")
            ax.axvline(P_27, color="0.3", lw=0.8, ls="--")
            ax.set_xlim(0.0, 0.65)
            ax.set_title(f"n = {n}")
            ax.set_xlabel("p̂")
        axes[0].legend(fontsize=8)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

### Ex M13a.28 — L'entropie et l'indice de Gini sur les lois à trois issues : lignes de niveau et maximum au centre 🎨 ★ ⏱️ 25 min
**Objectif :** dessiner l'entropie et l'indice de Gini sur le triangle des lois à trois issues, et voir où et comment ils sont maximaux.  
**Prérequis :** fiche §6 · **Parcours :** C, E

Une loi à trois issues $(p_1, p_2, p_3)$, positive et de somme 1, est un point d'un triangle : chaque sommet est une loi certaine, le milieu d'un côté une loi qui partage sa masse entre deux issues, le centre la loi uniforme. La cellule fournie donne `simplex_grid()`, une grille de lois (une par ligne, de coordonnées multiples de $\frac{1}{60}$), et `to_plane(P)`, qui place chaque loi dans le triangle. Écris :
- `entropy_bits(P)` : l'entropie en bits de chaque loi (une par ligne), $-\sum_k p_k\log_2 p_k$, où un terme $0 \log_2 0$ compte pour 0 **sans** calculer $\log_2 0$ (aucun avertissement) ;
- `gini(P)` : l'indice de Gini $1 - \sum_k p_k^2$ de chaque loi ;
- `draw_impurities(axes)` : dans les deux axes reçus (sans en ouvrir d'autres), le triangle et les lignes de niveau de l'entropie (à gauche) et de l'indice de Gini (à droite) sur `simplex_grid()`, avec `ax.tricontourf` ou `ax.tricontour` sur les points de `to_plane`, et une marque à la loi où chacun est le plus grand. Mets `ax.set_aspect("equal")` et un titre par axe.

La cellule de vérification contrôle tes valeurs sur cinq lois (comparées à `scipy.stats.entropy`), la symétrie des deux impuretés quand on permute les issues, la position de leur maximum, leur concavité sur 200 couples de lois, puis trace ta figure.

Dans ta copie : où est le maximum de chaque impureté, et pour quelle loi ? Pourquoi les lignes de niveau de Gini sont-elles des cercles centrés au centre du triangle (développe $\lVert \mathbf{p} - \mathbf{u} \rVert^2$, avec $\mathbf{u}$ la loi uniforme) ? Celles de l'entropie en sont-elles ? Sur un côté du triangle, une issue a la probabilité 0 : quelle fonction d'une seule variable retrouves-tu ? Près d'un sommet, quelle impureté monte le plus vite, et pourquoi (regarde la pente de $-p\log_2 p$ en 0) ?

In [ ]:
def simplex_grid(n=60):
    """The laws (p1, p2, p3) whose coordinates are multiples of 1/n, one law per row."""
    i, j = np.meshgrid(np.arange(n + 1), np.arange(n + 1), indexing="ij")
    keep = i + j <= n
    return np.column_stack([i[keep], j[keep], n - i[keep] - j[keep]]) / n


def to_plane(P):
    """The points of the plane of the laws P (one per row): the laws (1, 0, 0), (0, 1, 0) and (0, 0, 1) go to the
    vertices (0, 0), (1, 0) and (1/2, √3/2) of an equilateral triangle."""
    P = np.asarray(P, dtype=float)
    return P[:, 1] + P[:, 2] / 2, P[:, 2] * math.sqrt(3) / 2

In [ ]:
def entropy_bits(P):
    """The entropies in bits of the laws P (one per row), where a term 0 log₂ 0 counts as 0: an array."""
    raise NotImplementedError("−Σ p log₂ p sur chaque ligne, sans jamais calculer log₂ 0")


def gini(P):
    """The Gini indices 1 − Σ p_k² of the laws P (one per row): an array."""
    raise NotImplementedError("1 − la somme des carrés de chaque ligne")


def draw_impurities(axes):
    """In the two axes received (and no other): the triangle of the laws with three outcomes, the level lines of the
    entropy (left) and of the Gini index (right) on simplex_grid(), and a marker at the law where each is largest."""
    raise NotImplementedError("to_plane(simplex_grid()), ax.tricontourf, puis ax.plot pour le triangle et le maximum")

In [ ]:
with checked("M13a.28"):
    from scipy.stats import entropy as scipy_entropy

    LAWS_28 = np.array([[0.6, 0.3, 0.1], [0.0, 0.75, 0.25], [1.0, 0.0, 0.0], [0.2, 0.2, 0.6], [0.45, 0.1, 0.45]])
    GRID_28 = simplex_grid()
    results_28 = []

    def report_28(ok, success, failure):
        results_28.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    with warnings.catch_warnings(record=True) as caught_h:
        warnings.simplefilter("always")
        h_laws, h_grid = np.asarray(entropy_bits(LAWS_28), dtype=float), np.asarray(entropy_bits(GRID_28), dtype=float)
    with warnings.catch_warnings(record=True) as caught_g:
        warnings.simplefilter("always")
        g_laws, g_grid = np.asarray(gini(LAWS_28), dtype=float), np.asarray(gini(GRID_28), dtype=float)
    h_ok_28 = bool(h_laws.shape == (5,) and np.allclose(h_laws, scipy_entropy(LAWS_28.T, base=2), rtol=0, atol=1e-12)
                   and h_grid.shape == (len(GRID_28),) and not caught_h)
    g_ok_28 = bool(g_laws.shape == (5,) and np.allclose(g_laws, 1 - np.sum(LAWS_28 ** 2, axis=1), rtol=0, atol=1e-12)
                   and g_grid.shape == (len(GRID_28),) and not caught_g)
    report_28(h_ok_28, "entropy_bits : une entropie par loi, en bits, 0 log₂ 0 compris, sans avertissement",
              "entropy_bits : une entropie par LIGNE, en bits (log₂), où un terme 0 log₂ 0 compte pour 0 sans calculer "
              "log₂ 0 (np.where ou un masque), sans avertissement")
    report_28(g_ok_28, "gini : un indice de Gini par loi", "gini : 1 − la somme des carrés, une valeur par LIGNE")
    right_28 = [(f, values) for f, values, ok in ((entropy_bits, h_grid, h_ok_28), (gini, g_grid, g_ok_28)) if ok]
    if len(right_28) < 2:                            # the properties below are checked on right values only
        results_28.append(False)
        print("ℹ️ la symétrie, le maximum et la concavité se vérifient une fois les valeurs des deux impuretés justes")
    if right_28:
        swaps_28 = [[1, 2, 0], [1, 0, 2]]            # a rotation and a reflection of the triangle
        report_28(all(np.allclose(np.asarray(f(GRID_28[:, s]), dtype=float), values, rtol=0, atol=1e-12)
                      for f, values in right_28 for s in swaps_28),
                  "les impuretés ne dépendent pas de l'ordre des issues : leurs lignes de niveau ont les symétries du "
                  "triangle",
                  "une impureté change quand on permute les issues : elle doit traiter les trois de la même façon")
        center_28 = np.full(3, 1 / 3)
        report_28(all(np.allclose(GRID_28[np.argmax(values)], center_28) for _, values in right_28),
                  "le maximum est au centre du triangle, la loi uniforme",
                  "le maximum de tes impuretés n'est pas au centre du triangle : relis tes formules")
        pairs_28 = np.random.default_rng(4128).dirichlet(np.ones(3), size=(200, 2))
        middle_28 = pairs_28.mean(axis=1)
        report_28(all(np.all(np.asarray(f(middle_28), dtype=float)
                             >= (np.asarray(f(pairs_28[:, 0]), dtype=float) + np.asarray(f(pairs_28[:, 1]), dtype=float)) / 2
                             - 1e-12) for f, _ in right_28),
                  "concaves : l'impureté d'un mélange de deux lois est au moins la moyenne de leurs impuretés",
                  "une de tes impuretés n'est pas concave sur 200 couples de lois : relis tes formules")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    try:
        opened_28 = drawn(draw_impurities, axes)
    except BaseException:
        plt.close(fig)
        raise
    report_28(not opened_28 and all(ax.collections or ax.lines for ax in axes),
              "draw_impurities dessine dans les deux axes reçus",
              "draw_impurities doit dessiner dans les deux axes reçus, sans ouvrir de figure (ni plt.figure ni "
              "plt.subplots)")
    plt.show()
    verdict("M13a.28", all(results_28), "les impuretés et leur figure sont justes.",
            "relis les lignes ❌ ci-dessus.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1 à §3. Quatre erreurs à trouver dans un résumé de modèle, deux maxima de vraisemblance vérifiés avec SymPy, puis un maximum de vraisemblance sans formule.

### Ex M13a.29 — Chasse à l'erreur : un ln 0, une sigmoïde qui déborde, une variance de maximum de vraisemblance divisée par n − 1 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans un code de résumé d'un modèle les erreurs classiques de la vraisemblance et des scores, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1 à §3 · **Parcours :** C

Un collègue résume les sorties de deux modèles et un échantillon (la cellule fournie) : les probabilités `P_29` (arrondies : certaines valent exactement 0 ou 1), les scores `Z_29` (dont deux immenses) et des mesures `X_29`. Sept lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Quatre lignes ne font pas ce que dit leur commentaire. **Sans exécuter son code**, trouve-les.

```python
y, p, z, x = Y_29, P_29, Z_29, X_29
ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))       # (A) the log-likelihood of the labels y under p
prob = 1 / (1 + np.exp(-z))                                 # (B) σ(z), without any warning, even for z = −900
loss = np.mean(np.log(1 + np.exp(z)) - y * z)               # (C) the mean logistic loss of the scores z
p_hat = np.mean(y)                                          # (D) the maximum likelihood estimate of P(Y = 1)
base = -(p_hat * np.log(p_hat) + (1 - p_hat) * np.log(1 - p_hat))   # (E) the log-loss of the constant model p̂
mu = np.mean(x)                                             # (F) the maximum likelihood estimate of μ
var = np.var(x, ddof=1)                                     # (G) the maximum likelihood estimate of σ²
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) la bonne valeur de `ll` (4 décimales) ;
- c) la bonne valeur de `loss` (4 décimales) ;
- d) la bonne valeur de `var` (4 décimales).

Pour b) à d), écris le code corrigé dans la cellule (c'est permis : seule la recherche des lignes fausses se fait sans exécuter).

In [ ]:
Y_29 = np.array([1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0])                                  # 12 labels
P_29 = np.array([0.9, 0.0, 0.25, 1.0, 0.7, 0.4, 0.05, 0.55, 0.0, 0.3, 0.85, 0.15])           # a model's P(Y = 1), rounded
Z_29 = np.array([2.1, -900.0, -1.3, 900.0, 0.8, -0.2, -3.1, 0.3, -5.0, -0.9, 1.7, -2.4])    # another model's scores
X_29 = np.round(np.random.default_rng(4129).normal(5.0, 2.0, size=12), 2)                   # 12 measures

In [ ]:
answer_M13a_29a = ...  # a) letters, as text
answer_M13a_29b = ...  # b) a number (4 decimals)
answer_M13a_29c = ...  # c) a number (4 decimals)
answer_M13a_29d = ...  # d) a number (4 decimals)

In [ ]:
for letter, answer in zip("abcd", [answer_M13a_29a, answer_M13a_29b, answer_M13a_29c, answer_M13a_29d]):
    wbm.check(f"M13a.29{letter}", answer)

**Après la vérification**, dans ta copie : pour chaque ligne fausse, que se passe-t-il exactement (une valeur, `nan`, `inf`, un avertissement), et comment la corriger ? Laquelle donne une valeur juste mais avec un avertissement ? Laquelle donne un nombre faux sans aucun signal ? Pourquoi la ligne (E), qui ressemble à (A), est-elle juste ici ?

### Ex M13a.30 — Vérifier avec SymPy le maximum de vraisemblance d'une loi géométrique et d'une loi de Poisson 🛠️ ★ ⏱️ 15 min
**Objectif :** faire calculer par SymPy une log-vraisemblance, sa dérivée, son point critique et le signe de sa dérivée seconde, pour vérifier un maximum de vraisemblance fait à la main.  
**Prérequis :** fiche §1 ; M05a · **Parcours :** C

Vérifier un calcul à la main avec un logiciel de calcul formel, c'est un geste de métier. La cellule fournie définit les symboles `p` et `lam` (positifs) et deux échantillons :
- `GEOM_30`, cinq observations d'une loi **géométrique** (le rang du premier succès, $P(X = k) = (1 - p)^{k - 1}p$ pour $k \ge 1$) ;
- `POIS_30`, six observations d'une loi de **Poisson** ($P(X = k) = e^{-\lambda}\frac{\lambda^k}{k!}$ pour $k \ge 0$).

Fais d'abord le calcul à la main, dans ta copie : écris $\ell(p)$, dérive, résous $\ell'(p) = 0$ (tu dois trouver $\hat{p} = 1/\bar{x}$), et le signe de $\ell''$ ; de même pour $\lambda$ (tu dois trouver $\hat{\lambda} = \bar{x}$). Puis fais-le faire par SymPy, en définissant :
- `ell_geom` : $\ell(p)$ pour `GEOM_30`, une expression SymPy (une somme de `sympy.log(...)` sur les observations) ;
- `dell_geom` : sa dérivée, par `sympy.diff` ;
- `p_hat` : la solution de $\ell'(p) = 0$, par `sympy.solve` (un nombre SymPy, pas la liste que rend `solve`) ;
- `d2_geom` : $\ell''(\hat{p})$, la dérivée seconde en $\hat{p}$ (`sympy.diff(..., p, 2)`, puis `.subs(p, p_hat)`) ;
- `ell_pois`, `lam_hat` et `d2_pois` : la même chose pour `POIS_30` et $\lambda$ (les termes constants $-\ln x_i!$ peuvent être omis : ils ne changent ni la dérivée ni le maximum).

La cellule de vérification compare chacun de tes objets à un calcul indépendant (en quelques points pour les expressions) et dit ✅ ou ❌.

Dans ta copie : pourquoi le signe de $\ell''(\hat{p})$ suffit-il à conclure à un maximum ici (regarde le signe de $\ell''(p)$ sur tout $]0, 1[$) ? `sympy.solve` a-t-il trouvé d'autres solutions, et pourquoi pas ? Écris $\hat{p}$ et $\hat{\lambda}$ en fonction de $\bar{x}$ : lequel des deux est une moyenne, lequel l'inverse d'une moyenne, et pourquoi est-ce naturel (l'espérance d'une loi géométrique est $\frac{1}{p}$) ?

In [ ]:
p = sympy.symbols("p", positive=True)
lam = sympy.symbols("lambda", positive=True)
GEOM_30 = [3, 1, 4, 2, 5]        # observations of a geometric law: P(X = k) = (1 − p)^(k − 1) p, k = 1, 2, ...
POIS_30 = [2, 0, 3, 1, 4, 2]     # observations of a Poisson law: P(X = k) = e^(−λ) λ^k / k!, k = 0, 1, ...

In [ ]:
ell_geom = ...    # ℓ(p) for GEOM_30: a SymPy expression in p
dell_geom = ...   # ℓ′(p), with sympy.diff
p_hat = ...       # the solution of ℓ′(p) = 0, with sympy.solve: one SymPy number, not a list (SympifyError: [...] means a list)
d2_geom = ...     # ℓ″(p̂): a SymPy number
ell_pois = ...    # ℓ(λ) for POIS_30, a SymPy expression in lam (the constants −ln(xᵢ!) may be left out)
lam_hat = ...     # the solution of ℓ′(λ) = 0
d2_pois = ...     # ℓ″(λ̂)

In [ ]:
with checked("M13a.30"):
    def at_30(expr, symbol, point):
        """The value of the SymPy expression expr at symbol = point, as a float (None if not a real number)."""
        try:
            value = complex(sympy.sympify(expr).subs(symbol, point).evalf())
        except (TypeError, ValueError, AttributeError, sympy.SympifyError):
            return None
        return value.real if abs(value.imag) < 1e-12 else None

    def same_30(expr, reference, symbol, points, constant=False):
        """True when expr equals reference at the points (up to an additive constant if constant)."""
        gaps = []
        for point in points:
            a, b = at_30(expr, symbol, point), at_30(reference, symbol, point)
            if a is None or b is None:
                return False
            gaps.append((a - b, max(1.0, abs(b))))
        shift = gaps[0][0] if constant else 0.0
        return all(abs(gap - shift) <= 1e-9 * size for gap, size in gaps)

    def number_30(expr, expected):
        """True when expr is one number equal to expected; else False, or the reason as a text."""
        if isinstance(expr, (list, tuple, set, dict)):
            return "sympy.solve rend une LISTE de solutions : prends-en l'élément, sympy.solve(...)[0]"
        try:
            free = sympy.sympify(expr).free_symbols
            if free:
                return (f"c'est encore une expression de {', '.join(sorted(map(str, free)))} : remplace la variable "
                        f"par la solution, avec .subs(...)")
            return abs(float(sympy.sympify(expr)) - expected) <= 1e-9 * max(1.0, abs(expected))
        except (TypeError, ValueError, AttributeError, sympy.SympifyError):
            return False

    ref_geom_30 = sum((k - 1) * sympy.log(1 - p) + sympy.log(p) for k in GEOM_30)
    ref_pois_30 = sum(x * sympy.log(lam) - lam for x in POIS_30)
    items_30 = [
        ("ell_geom", lambda: ell_geom, lambda e: same_30(e, ref_geom_30, p, [0.2, 0.45, 0.8]),
         "ℓ(p), la log-vraisemblance géométrique : une somme de ln((1 − p)^(kᵢ − 1) p) sur les cinq observations"),
        ("dell_geom", lambda: dell_geom, lambda e: same_30(e, sympy.diff(ref_geom_30, p), p, [0.2, 0.45, 0.8]),
         "ℓ′(p) : sympy.diff(ell_geom, p)"),
        ("p_hat", lambda: p_hat, lambda e: number_30(e, 1 / sum(GEOM_30) * len(GEOM_30)),
         "p̂ : la solution de ℓ′(p) = 0 (sympy.solve), dans ]0, 1["),
        ("d2_geom", lambda: d2_geom,
         lambda e: number_30(e, float(sympy.diff(ref_geom_30, p, 2).subs(p, sympy.Rational(len(GEOM_30), sum(GEOM_30))))),
         "ℓ″(p̂) : la dérivée seconde de ℓ, puis .subs(p, p_hat)"),
        ("ell_pois", lambda: ell_pois, lambda e: same_30(e, ref_pois_30, lam, [0.7, 2.0, 3.5], constant=True),
         "ℓ(λ), la log-vraisemblance de Poisson : une somme de ln(e^(−λ) λ^(xᵢ) / xᵢ!)"),
        ("lam_hat", lambda: lam_hat, lambda e: number_30(e, sum(POIS_30) / len(POIS_30)),
         "λ̂ : la solution de ℓ′(λ) = 0"),
        ("d2_pois", lambda: d2_pois, lambda e: number_30(e, -sum(POIS_30) / (sum(POIS_30) / len(POIS_30)) ** 2),
         "ℓ″(λ̂) : la dérivée seconde de ℓ en λ̂"),
    ]
    done_30, right_30 = 0, 0
    for name, get, test, what in items_30:
        value = get()
        if not filled(value):
            print(f"⏳ {name} : pas encore écrit")
            continue
        done_30 += 1
        result = test(value)
        ok = result is True
        right_30 += ok
        print(f"✅ {name}" if ok else f"❌ {name} : {result if isinstance(result, str) else what}")
    if done_30 < len(items_30):
        print(f"⏳ Ex M13a.30 : {done_30} objets sur {len(items_30)} écrits, dont {right_30} justes.")
    else:
        verdict("M13a.30", right_30 == len(items_30),
                "SymPy confirme les deux maxima de vraisemblance, et le signe de la dérivée seconde dit que ce sont "
                "des maxima.", f"{len(items_30) - right_30} objet(s) faux : relis les lignes ❌ ci-dessus.")

### Ex M13a.31 — Défi : la proportion d'un mélange de deux lois connues par maximum de vraisemblance, à 1e−6 près du maximum (dichotomie ou Newton) 🏆 ★★★ ⏱️ 25 min
**Objectif :** trouver numériquement un maximum de vraisemblance qui n'a pas de formule, avec une méthode qui garantit sa précision et un budget d'évaluations.  
**Prérequis :** fiche §1 ; M05a ; M10 · **Parcours :** C

Des mesures viennent de deux sources mélangées, de densités **connues** $f_0 = \mathcal{N}(0, 1)$ et $f_1 = \mathcal{N}(2{,}5 ;\ 1)$, dans une proportion $\pi$ **inconnue** : chaque mesure suit la densité $\pi f_1(x) + (1 - \pi) f_0(x)$. Les 400 mesures `X_31` (cellule fournie) donnent la log-vraisemblance et sa dérivée

$$\ell(\pi) = \sum_{i=1}^{400} \ln\big(\pi f_1(x_i) + (1 - \pi) f_0(x_i)\big), \qquad \ell'(\pi) = \sum_{i=1}^{400} \frac{f_1(x_i) - f_0(x_i)}{\pi f_1(x_i) + (1 - \pi) f_0(x_i)},$$

calculées par `ell_31` et `dell_31`. Annuler $\ell'$ ne donne pas de formule : il faut chercher le maximum numériquement. Bonne nouvelle, $\ell$ est **concave** sur $[0, 1]$ (calcule $\ell''$ : une somme de carrés changée de signe), donc $\ell'$ décroît, et un point où elle s'annule est le maximum.

**L'objectif** : écrire `find_weight(ell, dell)`, qui reçoit deux fonctions d'un seul nombre $\pi$ (comme `ell_31` et `dell_31`) et rend le $\hat{\pi} \in [0, 1]$ qui maximise `ell`, à $10^{-6}$ près du vrai maximum, en appelant `ell` et `dell` **au plus 200 fois** en tout. La vérification compte les appels, et essaie ta fonction sur `X_31`, sur un autre échantillon, d'une autre proportion, puis sur deux fonctions concaves dont le maximum est au bord, en 0 et en 1 : ta fonction ne doit utiliser que les deux fonctions reçues (ni `X_31`, ni `ell_31`, ni `make_loglik`). Une grille unique ne suffit pas : $10^{-6}$ près demanderait un million de points. `scipy.optimize` (`minimize`, `brentq`, `fsolve`…), `np.polyfit` et scikit-learn sont interdits. Pense à la **dichotomie** sur $\ell'$ (M05a) ou à la méthode de **Newton** (une dérivée seconde approchée par les valeurs de `dell`), et au cas où le maximum serait au bord, en 0 ou en 1.

Une dernière cellule enregistre si l'objectif est atteint.

Dans ta copie : combien d'évaluations ta méthode a-t-elle utilisées, et combien en faudrait-il, au minimum, pour une précision de $10^{-12}$ ? Pourquoi la concavité garantit-elle que ta méthode ne s'arrête pas sur un faux maximum ? Ton $\hat{\pi}$ est-il égal à la proportion de points venus de $f_1$, et pourquoi pas exactement ?

In [ ]:
def normal_pdf(x, mu, sigma):
    """The density of N(mu, sigma²) at x."""
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * math.sqrt(2 * math.pi))


def mixture_sample(seed, weight, n=400):
    """n points of the mixture weight · N(2.5, 1) + (1 − weight) · N(0, 1)."""
    rng = np.random.default_rng(seed)
    from_f1 = rng.random(n) < weight
    return np.where(from_f1, rng.normal(2.5, 1.0, n), rng.normal(0.0, 1.0, n))


def make_loglik(x):
    """(ell, dell): the log-likelihood ℓ(π) of the mixture π f₁ + (1 − π) f₀ (f₁ = N(2.5, 1), f₀ = N(0, 1)) for the
    sample x, and its derivative ℓ′(π). Both take one number π of [0, 1]."""
    x = np.asarray(x, dtype=float)
    f0, f1 = normal_pdf(x, 0.0, 1.0), normal_pdf(x, 2.5, 1.0)

    def ell(pi):
        return float(np.sum(np.log(pi * f1 + (1 - pi) * f0)))

    def dell(pi):
        return float(np.sum((f1 - f0) / (pi * f1 + (1 - pi) * f0)))

    return ell, dell


X_31 = mixture_sample(4131, 0.35)                # 400 points; you know f₀ and f₁, not the proportion π
ell_31, dell_31 = make_loglik(X_31)
MAX_CALLS_31 = 200

In [ ]:
def find_weight(ell, dell):
    """The π in [0, 1] that maximizes ell (a concave function), to 1e-6, with at most MAX_CALLS_31 calls of ell and
    dell together; dell is the derivative of ell. Both take one number π."""
    raise NotImplementedError("dichotomie sur le signe de dell, ou méthode de Newton, en au plus 200 appels")

In [ ]:
with checked("M13a.31 défi"):
    import inspect
    import re

    from scipy.optimize import brentq                # the reference maximum (forbidden in find_weight)

    problems_31, ok_31, ran_31 = [], False, False
    try:
        source_31 = inspect.getsource(find_weight)
    except (OSError, TypeError):
        source_31 = ""
    banned_31 = sorted(set(re.findall(r"\b(scipy|optimize|minimize|minimize_scalar|brentq|fsolve|polyfit|sklearn|"
                                      r"X_31|ell_31|dell_31|make_loglik|normal_pdf|mixture_sample)\b", source_31)))
    if banned_31:
        ran_31 = True
        problems_31.append(f"find_weight utilise {', '.join(banned_31)}, interdit dans ce défi")
    cases_31 = [("l'échantillon X_31", *make_loglik(X_31)),
                ("un autre échantillon, d'une autre proportion", *make_loglik(mixture_sample(4132, 0.6))),
                ("une fonction concave de maximum au bord, en 0", lambda t: -(t + 0.3) ** 2, lambda t: -2 * (t + 0.3)),
                ("une fonction concave de maximum au bord, en 1", lambda t: -(t - 1.4) ** 2, lambda t: -2 * (t - 1.4))]
    for name_t, ell_t, dell_t in ([] if banned_31 else cases_31):
        if dell_t(0.0) <= 0:
            best_t = 0.0
        elif dell_t(1.0) >= 0:
            best_t = 1.0
        else:
            best_t = brentq(dell_t, 0.0, 1.0, xtol=1e-15, rtol=1e-15)
        calls_t = [0]

        def counted(function, calls=calls_t):
            def wrapper(pi):
                if np.ndim(pi) != 0:
                    raise TypeError("ell et dell prennent un seul nombre π, pas un tableau")
                calls[0] += 1
                return function(float(pi))
            return wrapper

        try:
            pi_t = find_weight(counted(ell_t), counted(dell_t))
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            ran_31 = True
            problems_31.append(f"find_weight lève {type(exc).__name__} ({exc})")
            break
        ran_31 = True
        try:
            pi_t = float(pi_t)
        except (TypeError, ValueError):
            problems_31.append(f"find_weight doit rendre un nombre, pas {type(pi_t).__name__}")
            break
        gap_t = abs(pi_t - best_t)
        print(f"{name_t} : {calls_t[0]} évaluation{'s' if calls_t[0] > 1 else ''} de ell et dell, "
              f"écart au maximum {gap_t:.1e}".replace(".", ","))
        if calls_t[0] == 0:
            problems_31.append(f"{name_t} : ta fonction n'appelle ni ell ni dell, alors qu'elle ne doit utiliser que "
                               "ces deux fonctions")
        if calls_t[0] > MAX_CALLS_31:
            problems_31.append(f"{name_t} : {calls_t[0]} évaluations, au-dessus du budget de {MAX_CALLS_31}")
        if not gap_t <= 1e-6:
            problems_31.append(f"{name_t} : ton π̂ est à {gap_t:.1e} du maximum, plus que 1e−6".replace(".", ","))
        if problems_31:
            break
    if problems_31:
        print("❌ Ex M13a.31 : " + " ; ".join(problems_31) + ".")
    elif ran_31:
        ok_31 = True
        print("✅ Ex M13a.31 : défi réussi : le maximum à 1e−6 près, en au plus 200 évaluations, sur les deux échantillons "
              "et sur deux fonctions de maximum au bord.")
with checked("M13a.31"):
    if not ran_31:                               # ⏳ while find_weight is not written
        raise NotImplementedError("find_weight n'est pas encore écrite")
    wbm.check("M13a.31", bool(ok_31), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire une log-vraisemblance et trouver son maximum pour une loi de Bernoulli, catégorielle et normale (et dire pourquoi la variance divise par $n$) ?
2. Sais-tu passer de la log-vraisemblance des labels à la log-loss, et la calculer à partir des scores sans débordement ?
3. Sais-tu lire un modèle logistique : probabilité, cote, logit, frontière ?
4. Sais-tu calculer un a posteriori de Naïve Bayes et dire ce que change l'hypothèse naïve ?
5. Sais-tu calculer une entropie, un indice de Gini, un gain d'information et une réduction de variance ?

**La suite** : M13b, optimisation sous contraintes et SVM, puis le chapitre 13 de workbookIA (classifieurs).